In [ ]:
# General notebook settings
import logging
import warnings

import pypsa

warnings.filterwarnings("error", category=DeprecationWarning)
# pandas<3.0.3 sets the `locs` attribute deprecated in matplotlib>=3.11
warnings.filterwarnings("ignore", message="The locs attribute was deprecated")
logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False

# Data Centre Investment with Willingness to Pay

This example shows how to model a new flexible consumer whose size is part of the investment decision, here a data centre. The data centre is modelled as an active, extendable [`Load`][pypsa.components.Loads]:

- `p_nom_extendable=True` lets the optimisation choose its capacity, paying `capital_cost` per MW.
- `marginal_cost` is the price the operator is willing to pay for electricity. For loads, the marginal cost applies to the power injected at the bus, which is negative for consumption, so every MWh consumed reduces the objective by the willingness to pay.
- `p_min_pu` sets a minimum utilisation of the installed capacity.

Since no `p_set` is given, the consumption of the data centre is a decision variable, while the regular demand with a fixed `p_set` stays a constant in the energy balance.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Network

We model one week with hourly resolution, scaled to represent a full year through the snapshot weightings. The system has a fixed demand of 100 MW and can invest in wind, solar, a battery and a gas turbine. The wind and solar availability profiles are synthetic.

In [ ]:
hours = 168
snapshots = pd.date_range("2030-01-01", periods=hours, freq="h")
rng = np.random.default_rng(42)

t = np.arange(hours)
solar_cf = np.clip(np.sin((t % 24 - 6) / 12 * np.pi), 0, None) * 0.8
wind_cf = np.clip(0.45 + 0.3 * np.sin(t / 30) + 0.1 * rng.standard_normal(hours), 0, 1)

n = pypsa.Network()
n.set_snapshots(snapshots)
n.snapshot_weightings.loc[:, :] = 8760 / hours

n.add("Carrier", ["AC", "solar", "wind", "battery", "gas", "demand", "data centre"])
n.add("Bus", "electricity", carrier="AC")
n.add("Load", "demand", bus="electricity", p_set=100, carrier="demand")

n.add(
    "Generator",
    "solar",
    bus="electricity",
    carrier="solar",
    p_nom_extendable=True,
    p_max_pu=solar_cf,
    capital_cost=40_000,  # EUR/MW/a
)
n.add(
    "Generator",
    "wind",
    bus="electricity",
    carrier="wind",
    p_nom_extendable=True,
    p_max_pu=wind_cf,
    capital_cost=100_000,  # EUR/MW/a
)
n.add(
    "Generator",
    "gas",
    bus="electricity",
    carrier="gas",
    p_nom_extendable=True,
    capital_cost=50_000,  # EUR/MW/a
    marginal_cost=120,  # EUR/MWh
)
n.add(
    "Store",
    "battery",
    bus="electricity",
    carrier="battery",
    e_nom_extendable=True,
    max_hours=4,
    efficiency_store=0.95,
    efficiency_dispatch=0.95,
    capital_cost=60_000,  # EUR/MWh/a
)

The data centre has annualised investment costs of 150,000 EUR/MW/a for its servers and building, needs to run at least at 50% of its capacity, and its site allows at most 200 MW. It is willing to pay up to 100 EUR/MWh for electricity.

The site limit `p_nom_max` matters: in a linear model, a consumer whose willingness to pay exceeds its costs at the margin would otherwise grow without bound.

In [ ]:
n.add(
    "Load",
    "data centre",
    bus="electricity",
    carrier="data centre",
    p_nom_extendable=True,
    p_nom_max=200,  # MW, site limit
    capital_cost=150_000,  # EUR/MW/a
    p_min_pu=0.5,
    marginal_cost=100,  # willingness to pay in EUR/MWh
)

n.optimize()

The optimisation builds the data centre, because the value of its consumption covers its investment costs and the additional system costs it causes. Within its operating range, its consumption follows the availability of cheap renewable electricity and avoids hours in which the gas turbine sets the price above its willingness to pay:

In [ ]:
n.c.loads.static.at["data centre", "p_nom_opt"]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
n.c.loads.dynamic.p.plot(ax=ax)
ax.set_ylabel("Consumption [MW]")
ax.set_xlabel("")
ax.legend(title="")
fig.tight_layout()

In the statistics, the willingness to pay of the data centre shows up as negative operational expenditure, i.e. the utility of its consumption, next to its capital expenditure:

In [ ]:
n.statistics()[
    ["Optimal Capacity", "Withdrawal", "Capital Expenditure", "Operational Expenditure"]
]

## Sensitivity to the willingness to pay

How much the data centre values electricity determines whether it is built and how it runs. Since all costs scale linearly with capacity, the data centre is mostly either not built or built up to its site limit, with a narrow transition around the break-even willingness to pay, where the system costs it causes start to bind. Below break-even, it is not worth building. Above, its utilisation increases with the willingness to pay, until it exceeds the marginal cost of the gas turbine (120 EUR/MWh) and the data centre runs at full capacity.

In [ ]:
results = {}
for wtp in [30, 40, 45, 50, 55, 60, 80, 100, 120, 130, 160]:
    n.c.loads.static.loc["data centre", "marginal_cost"] = wtp
    n.optimize()
    capacity = n.c.loads.static.at["data centre", "p_nom_opt"]
    consumption = n.c.loads.dynamic.p["data centre"].mean()
    results[wtp] = {
        "capacity [MW]": capacity,
        "utilisation [-]": consumption / capacity if capacity > 1e-3 else 0.0,
    }

results = pd.DataFrame(results).T.rename_axis("willingness to pay [EUR/MWh]")
results.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
results["capacity [MW]"].plot(ax=axes[0], marker="o")
axes[0].set_ylabel("Data centre capacity [MW]")
results["utilisation [-]"].plot(ax=axes[1], marker="o")
axes[1].set_ylabel("Utilisation [-]")
fig.tight_layout()

!!! note "Generators with negative sign"

    Before loads became active components, the same problem was modelled with a `Generator` with `sign=-1` and a negative `marginal_cost`. `Load(marginal_cost=a)` is equivalent to `Generator(sign=-1, marginal_cost=-a)`. See the [Load component documentation](../user-guide/components/loads.md) for details.